In [6]:
#Khởi chạy các thư viện
#pip install ta vnstock yfinance lightgbm xgboost

In [7]:
#thư viện
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
#tắt cảnh báo
import warnings
warnings.filterwarnings('ignore')

In [9]:
#Thiết lập các dữ liệu trong dataframe chỉ hiện 3 số sau dấu phẩy
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Chỉ báo kỹ thuật

In [10]:
raw_data = pd.read_excel("VNINDEX-Forecasting-Data.xlsx", sheet_name="Dữ liệu thô", index_col=0)
raw_data = raw_data.dropna()

raw_data.head()

,open,high,low,close,volume,USD/VND,Gold,Oil,SP500
2015-01-05,544.860,549.220,543.780,544.450,91834620.000,21046.000,1203.900,50.040,2020.580
2015-01-06,539.080,550.110,538.820,549.660,94081890.000,21097.000,1219.300,47.930,2002.610
2015-01-07,548.440,555.830,548.440,552.050,109445780.000,21083.000,1210.600,48.650,2025.900
2015-01-08,553.490,556.800,552.150,553.470,73883040.000,21061.000,1208.400,48.790,2062.140
2015-01-09,553.490,570.520,552.150,569.730,104203680.000,21021.000,1216.000,48.360,2044.810


In [11]:
#thủ công
# Tính ROC (Rate of Change)
def calculate_roc(df, n=14):
    df['roc'] = ((df['close'] - df['close'].shift(n)) / df['close'].shift(n)) * 100
    return df

# Tính CCI (Commodity Channel Index)
def calculate_cci(df, n=20):
    df['typical_price'] = (df['high'] + df['low'] + df['close']) / 3
    df['sma'] = df['typical_price'].rolling(window=n).mean()
    df['mad'] = df['typical_price'].rolling(window=n).apply(lambda x: np.fabs(x - x.mean()).mean(), raw=True)
    df['cci'] = (df['typical_price'] - df['sma']) / (0.015 * df['mad'])
    return df

# Tính RSI (Relative Strength Index)
def calculate_rsi(df, n=14):
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=n).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=n).mean()

    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    return df

# Tính MACD (Moving Average Convergence Divergence)
def calculate_macd(df, fast=12, slow=26, signal=9):
    df['ema_fast'] = df['close'].ewm(span=fast, adjust=False).mean()
    df['ema_slow'] = df['close'].ewm(span=slow, adjust=False).mean()
    df['macd'] = df['ema_fast'] - df['ema_slow']
    df['macd_signal'] = df['macd'].ewm(span=signal, adjust=False).mean()
    df['macd_hist'] = df['macd'] - df['macd_signal']
    return df

# Tính OBV (On-balance Volume)
def calculate_obv(df):
    df['obv'] = np.where(df['close'] > df['close'].shift(1), df['volume'], 
                         np.where(df['close'] < df['close'].shift(1), -df['volume'], 0))
    df['obv'] = df['obv'].cumsum()  # Tích lũy OBV
    return df

indicators = raw_data.copy()

# Áp dụng các hàm tính toán
indicators = calculate_roc(indicators)
indicators = calculate_cci(indicators)
indicators = calculate_rsi(indicators)
indicators = calculate_macd(indicators)
indicators = calculate_obv(indicators)

# Xem kết quả
indicators

,open,high,low,close,volume,USD/VND,Gold,Oil,SP500,roc,...,sma,mad,cci,rsi,ema_fast,ema_slow,macd,macd_signal,macd_hist,obv
2015-01-05,544.860,549.220,543.780,544.450,91834620.000,21046.000,1203.900,50.040,2020.580,NaN,...,NaN,NaN,NaN,NaN,544.450,544.450,0.000,0.000,0.000,0.000
2015-01-06,539.080,550.110,538.820,549.660,94081890.000,21097.000,1219.300,47.930,2002.610,NaN,...,NaN,NaN,NaN,NaN,545.252,544.836,0.416,0.083,0.332,94081890.000
2015-01-07,548.440,555.830,548.440,552.050,109445780.000,21083.000,1210.600,48.650,2025.900,NaN,...,NaN,NaN,NaN,NaN,546.297,545.370,0.927,0.252,0.675,203527670.000
2015-01-08,553.490,556.800,552.150,553.470,73883040.000,21061.000,1208.400,48.790,2062.140,NaN,...,NaN,NaN,NaN,NaN,547.401,545.970,1.431,0.488,0.943,277410710.000
2015-01-09,553.490,570.520,552.150,569.730,104203680.000,21021.000,1216.000,48.360,2044.810,NaN,...,NaN,NaN,NaN,NaN,550.836,547.730,3.106,1.011,2.095,381614390.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-26,1333.330,1338.550,1323.690,1326.090,672071249.000,25600.000,3020.900,69.650,5712.200,0.597,...,1322.733,8.754,51.104,56.072,1325.597,1314.758,10.839,12.852,-2.013,94630904298.000
2025-03-27,1325.980,1328.820,1323.010,1323.810,522142993.000,25555.000,3060.200,69.920,5693.310,-0.169,...,1323.792,7.519,12.601,48.110,1325.322,1315.428,9.894,12.261,-2.367,94108761305.000
2025-03-28,1324.420,1325.340,1315.720,1317.460,601970197.000,25560.000,3086.500,69.360,5580.940,-0.964,...,1324.521,6.593,-50.700,39.557,1324.112,1315.579,8.534,11.515,-2.982,93506791108.000
2025-03-31,1313.510,1314.090,1304.100,1306.860,706353460.000,25550.000,3122.800,71.480,5611.850,-1.927,...,1324.548,6.561,-164.580,31.583,1321.458,1314.933,6.525,10.517,-3.992,92800437648.000


In [12]:
# Xóa cột 
data = indicators.drop(columns=['typical_price', 'sma', 'ema_fast', 'ema_slow', 'mad', 'macd_hist'])

In [13]:
#Hoàn chỉnh dữ liệu để đưa vào mô hình
with pd.ExcelWriter("VNINDEX-Forecasting-Data.xlsx", mode='a', engine='openpyxl') as writer: 
    data.to_excel(writer, sheet_name="Biến")